# Vine Copulas con optimizacion basado en CVaR





In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

tickers = [
    "AAPL",
    "MSFT",
    "AMZN",
    "GOOGL",
    "META"
]

prices = yf.download(
    tickers,
    start="2018-01-01",
    end="2025-01-01",
    auto_adjust=True
)["Close"]

returns = np.log(prices / prices.shift(1)).dropna()

[*********************100%***********************]  5 of 5 completed


# Ajustar marginales empiricas

In [2]:
from scipy.stats import rankdata

u = pd.DataFrame(index=returns.index)

for col in returns.columns:
    u[col] = rankdata(returns[col]) / (len(returns) + 1)

# Ajustar la Vine-copula

In [3]:
import pyvinecopulib as pv

u_np = u.values

vine = pv.Vinecop.from_data(u_np)

# Simular escenarios

In [4]:
n_scenarios = 50000

u_sim = vine.simulate(n_scenarios)

# reconstruccion de retornos reales

In [5]:
sim_returns = np.zeros_like(u_sim)

for j, col in enumerate(returns.columns):

    sorted_r = np.sort(returns[col].values)

    idx = np.floor(
        u_sim[:, j] * (len(sorted_r)-1)
    ).astype(int)

    sim_returns[:, j] = sorted_r[idx]

# Funcion CVAR

In [6]:
def cvar(portfolio_returns, alpha=0.95):

    var = np.quantile(portfolio_returns, 1 - alpha)

    tail = portfolio_returns[portfolio_returns <= var]

    return -np.mean(tail)

# Funcion objetivo
La literatura reciente suele usar:
maxE[R 
p
​	
 ]
sujeto a:
CVaR 
95%
​	
 (R 
p
​	
 )≤c
en lugar de maximizar el ratio Retorno/CVaR.
Con scipy.optimize sería:

In [7]:
def expected_return(weights):
    return np.mean(sim_returns @ weights)

In [8]:
cvar_limit = 0.03  # 3% pérdida esperada en cola

def cvar_constraint(weights):
    port = sim_returns @ weights
    return cvar_limit - cvar(port)

In [9]:
def sum_constraint(weights):
    return np.sum(weights) - 1

# Optimizacion

In [10]:
from scipy.optimize import minimize

n_assets = len(tickers)

bounds = [(0, 1)] * n_assets

constraints = [
    {"type": "eq", "fun": sum_constraint},
    {"type": "ineq", "fun": cvar_constraint}
]

w0 = np.ones(n_assets) / n_assets

result = minimize(
    fun=lambda w: -expected_return(w),
    x0=w0,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

weights_opt = result.x

# Resultados

In [11]:
weights = pd.Series(weights_opt, index=tickers)
print(weights.sort_values(ascending=False))

META     4.312003e-01
AAPL     3.235680e-01
AMZN     1.890213e-01
MSFT     5.621034e-02
GOOGL    4.176962e-14
dtype: float64
